# Manual Upload Preprocessing

Normalize heterogeneous CSV files in `local_data/manual_upload/` into the pipeline extraction schema, then create a standardized manual dataset in `local_data/manual/`.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path("src").resolve()))

from etl_pipeline.common.io import read_csv, write_csv, write_json
from etl_pipeline.extraction.manual_upload import run_manual_upload_extraction
from etl_pipeline.preprocessing.schema import STANDARDIZED_FIELDS
from etl_pipeline.preprocessing.transform import transform_extraction_rows

CONFIG = {
    "paths": {
        "local_data_dir": "local_data",
        "master_dir": "local_data/master",
        "logs_dir": "logs",
    },
    "manual_upload": {
        "input_dir": "local_data/manual_upload",
        "output_file": "local_data/manual/manual_extraction_raw.csv",
    },
}
RUN_ID = "manual_upload_notebook"
MANUAL_DIR = Path("local_data/manual")
STANDARDIZED_OUTPUT = MANUAL_DIR / "manual_standardized.csv"

In [ ]:
manual_summary = run_manual_upload_extraction(CONFIG, run_id=RUN_ID)
manual_summary

In [ ]:
raw_rows = read_csv(manual_summary["output_file"])
standardized_rows, transform_stats = transform_extraction_rows(raw_rows)

write_csv(STANDARDIZED_OUTPUT, standardized_rows, STANDARDIZED_FIELDS)
standardized_summary = {
    "stage": "manual_standardization",
    "input_file": manual_summary["output_file"],
    "output_file": str(STANDARDIZED_OUTPUT),
    "input_rows": transform_stats.input_rows,
    "output_rows": len(standardized_rows),
    "dropped_empty_text_rows": transform_stats.dropped_empty_text_rows,
    "deduplicated_rows": transform_stats.deduplicated_rows,
    "comments_exploded": transform_stats.comments_exploded,
    "replies_exploded": transform_stats.replies_exploded,
    "invalid_comments_json_rows": transform_stats.invalid_comments_json_rows,
}
write_json(MANUAL_DIR / "manual_standardized_summary.json", standardized_summary)
standardized_summary

In [ ]:
standardized_rows[:3]